## nb29 — CD5 Trajectory: Award vs. Control (±5 Years)

1. Load `author_papers_panel.csv` from nb28
2. Compute CD5 for every unique paper (reusing nb20 exact formula)
3. Aggregate to author x relative_year mean CD5
4. Plot event-study trajectory with 95% CI
5. DiD regression

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
import statsmodels.formula.api as smf

ROOT    = Path('..')
CD_DIR  = ROOT / 'data' / 'cd_trajectory'
FIG_DIR = ROOT / 'data' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

API_KEY = 'A08hCjeUoeVKA9toVsfCpF'

In [ ]:
panel = pd.read_csv(CD_DIR / 'author_papers_panel.csv')
print(panel.shape)
print(panel.groupby('group')['author_id'].nunique())

unique_papers = panel[['paper_id', 'paper_year']].drop_duplicates().reset_index(drop=True)
print(f'Unique papers to score: {len(unique_papers)}')

In [ ]:
def get_references(work_id):
    url = f'https://api.openalex.org/works/{work_id}'
    r = requests.get(url, params={'api_key': API_KEY}, timeout=15)
    if r.status_code != 200:
        return []
    refs = r.json().get('referenced_works', [])
    return [ref.split('/')[-1] for ref in refs]

def get_citations(work_id, focal_year, window=5):
    url = 'https://api.openalex.org/works'
    all_citers = []
    cursor = '*'
    while True:
        params = {
            'filter': f'cites:{work_id},publication_year:{focal_year}-{focal_year+window}',
            'per-page': 200,
            'select': 'id',
            'cursor': cursor,
            'api_key': API_KEY,
        }
        r = requests.get(url, params=params, timeout=15)
        if r.status_code != 200:
            break
        data = r.json()
        results = data.get('results', [])
        all_citers.extend([w['id'].split('/')[-1] for w in results])
        cursor = data.get('meta', {}).get('next_cursor')
        if not cursor or not results:
            break
        time.sleep(0.15)
    return all_citers

def get_refs_of_citer(citer_id):
    url = f'https://api.openalex.org/works/{citer_id}'
    r = requests.get(url, params={'api_key': API_KEY}, timeout=15)
    if r.status_code != 200:
        return set()
    refs = r.json().get('referenced_works', [])
    return set(ref.split('/')[-1] for ref in refs)

def compute_cd5(work_id, focal_year):
    focal_refs  = set(get_references(work_id))
    focal_cites = get_citations(work_id, focal_year)
    if not focal_cites:
        return np.nan, 0, 0
    ni, nj = 0, 0
    for citer in focal_cites:
        citer_refs = get_refs_of_citer(citer)
        if citer_refs & focal_refs:
            nj += 1
        else:
            ni += 1
        time.sleep(0.15)
    total = ni + nj
    cd = (ni - nj) / total if total > 0 else np.nan
    return cd, ni, nj

In [ ]:
CHECKPOINT = CD_DIR / 'paper_cd5_scores.csv'

if CHECKPOINT.exists():
    done_cd  = pd.read_csv(CHECKPOINT)
    done_ids = set(done_cd['paper_id'].unique())
    print(f'Resuming — {len(done_ids)} papers already scored')
else:
    done_cd  = pd.DataFrame()
    done_ids = set()

rows = []
todo = unique_papers[~unique_papers['paper_id'].isin(done_ids)].reset_index(drop=True)
print(f'Remaining: {len(todo)} papers')

for i, row in todo.iterrows():
    wid  = row['paper_id']
    year = int(row['paper_year'])
    cd, ni, nj = compute_cd5(wid, year)
    rows.append({'paper_id': wid, 'paper_year': year, 'cd5': cd, 'ni': ni, 'nj': nj})

    if (i + 1) % 50 == 0 or (i + 1) == len(todo):
        batch_df = pd.DataFrame(rows)
        done_cd  = pd.concat([done_cd, batch_df], ignore_index=True)
        done_cd.to_csv(CHECKPOINT, index=False)
        rows = []
        print(f'[{i+1}/{len(todo)}] saved — {done_cd["cd5"].notna().sum()} valid scores')

    time.sleep(0.2)

print('CD5 scoring done!')

In [ ]:
cd_scores = pd.read_csv(CD_DIR / 'paper_cd5_scores.csv')
panel_cd  = panel.merge(cd_scores[['paper_id', 'cd5']], on='paper_id', how='left')

print(f'Panel rows: {len(panel_cd)}')
print(f"CD5 coverage: {panel_cd['cd5'].notna().mean():.1%}")
panel_cd.head()

In [ ]:
author_year = (
    panel_cd
    .dropna(subset=['cd5'])
    .groupby(['author_id', 'group', 'award_year', 'relative_year'])['cd5']
    .mean()
    .reset_index()
)
print(author_year.shape)
author_year.head()

In [ ]:
from scipy import stats

def ci95(x):
    n = x.notna().sum()
    if n < 2: return np.nan
    se = stats.sem(x.dropna())
    return se * stats.t.ppf(0.975, df=n-1)

traj = (
    author_year
    .groupby(['group', 'relative_year'])['cd5']
    .agg(mean='mean', ci=ci95, n='count')
    .reset_index()
)
traj = traj[traj['relative_year'].between(-5, 5)]
print(traj)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

colors = {'award': '#01696f', 'control': '#964219'}
labels = {'award': 'Award authors', 'control': 'Control authors'}

for grp, gdf in traj.groupby('group'):
    gdf = gdf.sort_values('relative_year')
    ax.plot(gdf['relative_year'], gdf['mean'],
            marker='o', linewidth=2.2, color=colors[grp], label=labels[grp])
    ax.fill_between(gdf['relative_year'],
                    gdf['mean'] - gdf['ci'],
                    gdf['mean'] + gdf['ci'],
                    alpha=0.15, color=colors[grp])

ax.axvline(0, color='#28251d', linewidth=1.2, linestyle='--', alpha=0.6)
ax.axhline(0, color='#bab9b4', linewidth=0.8, linestyle=':')
ax.text(0.18, ax.get_ylim()[0] + 0.01, 'Award year', fontsize=9, color='#7a7974')

ax.set_xlabel('Years relative to award', fontsize=11)
ax.set_ylabel('Mean CD5 score', fontsize=11)
ax.set_title('Scientific Disruption (CD5) Trajectory\nAward vs. Control Authors (±5 Years)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(range(-5, 6))
ax.legend(frameon=False, fontsize=10)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
fig.savefig(FIG_DIR / 'cd5_trajectory_award_vs_control.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
import statsmodels.formula.api as smf

reg_df = author_year.copy()
reg_df['treated'] = (reg_df['group'] == 'award').astype(int)
reg_df['post']    = (reg_df['relative_year'] >= 0).astype(int)
reg_df['year_fe'] = reg_df['award_year'].astype(str)

model = smf.ols(
    'cd5 ~ treated * post + C(relative_year) + C(year_fe)',
    data=reg_df.dropna(subset=['cd5'])
).fit(cov_type='HC3')

print(model.summary().tables[1])

did_coef = model.params.get('treated:post', np.nan)
did_pval = model.pvalues.get('treated:post', np.nan)
print(f'\nDiD estimate (treated x post): {did_coef:.4f}  p={did_pval:.4f}')

In [ ]:
reg_df['rel_str'] = reg_df['relative_year'].astype(str)

es_model = smf.ols(
    'cd5 ~ treated:C(rel_str) + C(rel_str) + C(year_fe)',
    data=reg_df.dropna(subset=['cd5'])
).fit(cov_type='HC3')

es_coefs = {k: v for k, v in es_model.params.items() if 'treated:C(rel_str)' in k}
es_ses   = {k: v for k, v in es_model.bse.items()   if 'treated:C(rel_str)' in k}

es_df = pd.DataFrame({
    'rel_year': [int(k.split('T.')[-1].rstrip(']')) for k in es_coefs],
    'coef':     list(es_coefs.values()),
    'se':       list(es_ses.values()),
}).sort_values('rel_year')

fig2, ax2 = plt.subplots(figsize=(9, 4))
ax2.errorbar(es_df['rel_year'], es_df['coef'],
             yerr=1.96 * es_df['se'],
             fmt='o', color='#01696f', capsize=4, linewidth=1.8)
ax2.axhline(0, color='#bab9b4', linestyle='--')
ax2.axvline(0, color='#28251d', linewidth=1, linestyle=':', alpha=0.6)
ax2.set_xlabel('Years relative to award', fontsize=11)
ax2.set_ylabel('DiD coefficient (CD5)', fontsize=11)
ax2.set_title('Event-Study: Differential CD5 — Award vs. Control', fontsize=13, fontweight='bold')
ax2.set_xticks(range(-5, 6))
ax2.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig2.savefig(FIG_DIR / 'cd5_event_study_did.png', dpi=200, bbox_inches='tight')
plt.show()
print('Event-study plot saved.')

In [ ]:
author_year.to_csv(CD_DIR / 'author_year_cd5.csv', index=False)
print('author_year_cd5.csv saved')